# 02 Compute Metrics

Compute offline recommendation metrics and marketing-safe proxy KPIs for the DDM analytics layer.

This notebook does not train SR-GNN and does not create a new model split. It inherits the selected `v1_strict_filter` / `srgnn_fc` context from the backbone repo, including SR-GNN top-20 prediction rows exported from the already trained artifact.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs/project_config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

from ddm.io import load_config
from ddm.pipeline import inherit_recsys_context, compute_metrics_and_kpis


## Load Config

In [ ]:
config = load_config(PROJECT_ROOT / "configs/project_config.yaml")
config["inheritance"]

## Inherit SR-GNN Context

Required files are prepared under `data/inherited/recsys/v1_strict_filter_srgnn_fc_top20/`. `predictions.parquet` now contains SR-GNN top-20 rows exported from the already trained backbone artifact; no retraining or re-splitting is performed.


In [ ]:
inheritance = inherit_recsys_context(PROJECT_ROOT)
inheritance

## Validate Inherited Schemas

In [ ]:
inherited_root = PROJECT_ROOT / config["inheritance"]["inherited_root"]
test_examples = pd.read_parquet(inherited_root / "test_examples.parquet")
print(test_examples.shape)
test_examples.head()

In [ ]:
required_test_columns = {
    "example_id",
    "session_id",
    "target_item_id_internal",
    "target_item_id_raw",
    "eventdate",
}
missing = required_test_columns.difference(test_examples.columns)
assert not missing, f"Missing inherited test example columns: {sorted(missing)}"

predictions_path = inherited_root / "predictions.parquet"
if predictions_path.exists():
    predictions = pd.read_parquet(predictions_path)
    required_prediction_columns = {"example_id", "rank", "pred_item_id_internal", "pred_item_id_raw"}
    missing_predictions = required_prediction_columns.difference(predictions.columns)
    assert not missing_predictions, f"Missing prediction columns: {sorted(missing_predictions)}"
    print(predictions.shape)
    display(predictions.head())
else:
    todo_path = inherited_root / "PREDICTIONS_EXPORT_TODO.md"
    print("SR-GNN predictions.parquet is not available.")
    print(todo_path.read_text().splitlines()[0] if todo_path.exists() else "No TODO file found.")

## Compute Metrics and KPI Marts

The runner writes:

- `data/mart/fact_metrics.parquet`
- `data/mart/fact_marketing_kpis.parquet`
- `data/mart/fact_recommendations.parquet`
- `data/mart/fact_test_examples.parquet`
- `data/mart/fact_recommendation_eval.parquet`
- `data/mart/dim_model.parquet`

`fact_recommendations` contains long top-k rows for SR-GNN, popularity, and co-occurrence where available. `fact_recommendation_eval` is one row per model/example for session-level hit, rank, and value-proxy analysis.


In [ ]:
outputs = compute_metrics_and_kpis(PROJECT_ROOT)
outputs

## Offline Recommendation Metrics

In [ ]:
fact_metrics = pd.read_parquet(PROJECT_ROOT / "data/mart/fact_metrics.parquet")
fact_metrics.sort_values(["model_key", "metric_name"])

## Session-Centered Marketing Proxy KPIs

Safe wording:

- Recommendation Success Rate@20 is HR@20.
- CTR Proxy@20 is HR@20 in business language, not real CTR.
- GMV and purchase value fields are price-proxy based and offline only.
- Relative-delta rows are benchmark deltas, not causal revenue or conversion lift.

In [ ]:
fact_marketing_kpis = pd.read_parquet(PROJECT_ROOT / "data/mart/fact_marketing_kpis.parquet")
fact_marketing_kpis.sort_values(["model_key", "kpi_name"])

## Recommendation Rows

In [ ]:
fact_recommendations = pd.read_parquet(PROJECT_ROOT / "data/mart/fact_recommendations.parquet")
print(fact_recommendations.shape)
fact_recommendations.groupby("model_key").agg(
    rows=("example_id", "size"),
    examples=("example_id", "nunique"),
    unique_items=("pred_item_id_raw", "nunique"),
    min_rank=("rank", "min"),
    max_rank=("rank", "max"),
).reset_index()


## Per-Example Recommendation Evaluation


In [ ]:
fact_recommendation_eval = pd.read_parquet(PROJECT_ROOT / "data/mart/fact_recommendation_eval.parquet")
print(fact_recommendation_eval.shape)
fact_recommendation_eval.groupby("model_key").agg(
    rows=("example_id", "size"),
    examples=("example_id", "nunique"),
    hit_rate=("hit_at_k", "mean"),
    mean_reciprocal_rank=("reciprocal_rank", "mean"),
    captured_value_proxy=("captured_value_proxy", "sum"),
).reset_index()
